In [1]:
# Program 4: Candidate-Elimination Algorithm

# Same dataset used in Problem 3
data = [
    ["Sunny", "Warm", "Normal", "Strong", "Warm", "Same", "Yes"],
    ["Sunny", "Warm", "High", "Strong", "Warm", "Same", "Yes"],
    ["Rainy", "Cold", "High", "Strong", "Warm", "Change", "No"],
    ["Sunny", "Warm", "High", "Strong", "Cool", "Change", "Yes"]
]

# Number of attributes
num_attributes = len(data[0]) - 1

# Most Specific hypothesis
S = ["0"] * num_attributes

# Most General hypothesis
G = [["?"] * num_attributes]


# Function to check whether a hypothesis covers an example
def covers(hypothesis, example):

    for i in range(num_attributes):

        if hypothesis[i] != "?" and hypothesis[i] != example[i]:
            return False

    return True


# Process every training example
for example in data:

    attributes = example[:-1]
    target = example[-1]

    print("\nExample:", example)

    # -----------------------------------
    # POSITIVE EXAMPLE
    # -----------------------------------
    if target == "Yes":

        print("Positive Example")

        # Generalize S
        for i in range(num_attributes):

            if S[i] == "0":
                S[i] = attributes[i]

            elif S[i] != attributes[i]:
                S[i] = "?"

        # Remove hypotheses from G
        # that do not cover the positive example
        G = [g for g in G if covers(g, attributes)]


    # -----------------------------------
    # NEGATIVE EXAMPLE
    # -----------------------------------
    else:

        print("Negative Example")

        # Remove hypotheses from G
        # that cover the negative example
        new_G = []

        for g in G:

            if covers(g, attributes):

                # Specialize G
                for i in range(num_attributes):

                    if g[i] == "?":

                        # Use S value for specialization
                        if S[i] != "0" and S[i] != "?":

                            new_hypothesis = g.copy()

                            new_hypothesis[i] = S[i]

                            # Make sure it does not cover negative example
                            if not covers(new_hypothesis, attributes):
                                new_G.append(new_hypothesis)

            else:
                new_G.append(g)

        G = new_G


    # Display current boundaries
    print("S =", S)
    print("G =", G)


# -----------------------------------
# FINAL RESULT
# -----------------------------------

print("\n====================================")
print("FINAL CANDIDATE-ELIMINATION RESULT")
print("====================================")

print("\nMost Specific Hypothesis (S):")
print(S)

print("\nMost General Hypotheses (G):")

for hypothesis in G:
    print(hypothesis)


Example: ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same', 'Yes']
Positive Example
S = ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same']
G = [['?', '?', '?', '?', '?', '?']]

Example: ['Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same', 'Yes']
Positive Example
S = ['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']
G = [['?', '?', '?', '?', '?', '?']]

Example: ['Rainy', 'Cold', 'High', 'Strong', 'Warm', 'Change', 'No']
Negative Example
S = ['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']
G = [['Sunny', '?', '?', '?', '?', '?'], ['?', 'Warm', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', 'Same']]

Example: ['Sunny', 'Warm', 'High', 'Strong', 'Cool', 'Change', 'Yes']
Positive Example
S = ['Sunny', 'Warm', '?', 'Strong', '?', '?']
G = [['Sunny', '?', '?', '?', '?', '?'], ['?', 'Warm', '?', '?', '?', '?']]

FINAL CANDIDATE-ELIMINATION RESULT

Most Specific Hypothesis (S):
['Sunny', 'Warm', '?', 'Strong', '?', '?']

Most General Hypotheses (G):
['Sunny', '?', '?', '?', '?', '?']
['?'

In [1]:
import pandas as pd

# -----------------------------------
# 1. Load Dataset
# -----------------------------------
df = pd.read_csv("playtennis.csv")

print("Original Dataset:")
print(df)

print("\nDataset Shape:", df.shape)


# -----------------------------------
# 2. Remove rows with missing values
# -----------------------------------
df = df.dropna().reset_index(drop=True)

print("\nDataset after removing missing values:")
print(df)


# -----------------------------------
# 3. Identify attributes and target
# -----------------------------------
target_column = "PlayTennis"

attributes = [
    "Outlook",
    "Temperature",
    "Humidity",
    "Wind"
]

print("\nTarget:", target_column)
print("Attributes:", attributes)


# -----------------------------------
# 4. Get all possible attribute values
# -----------------------------------
domains = {}

for attribute in attributes:
    domains[attribute] = list(df[attribute].unique())

print("\nAttribute Domains:")
for attribute in attributes:
    print(attribute, ":", domains[attribute])


# -----------------------------------
# 5. Candidate-Elimination Algorithm
# -----------------------------------
def candidate_elimination(data, attributes, target, domains):

    # Most Specific Hypothesis
    S = ["Ø"] * len(attributes)

    # Most General Hypothesis
    G = [["?"] * len(attributes)]

    print("\n========================================")
    print("Candidate-Elimination Algorithm")
    print("========================================")

    # -----------------------------------
    # Process every training example
    # -----------------------------------
    for index, row in data.iterrows():

        example = [
            row[attribute]
            for attribute in attributes
        ]

        label = row[target].lower()

        print("\n----------------------------------------")
        print("Example", index + 1)
        print("Training Example:", example)
        print("Class:", row[target])

        # ===================================
        # POSITIVE EXAMPLE
        # ===================================
        if label == "yes":

            # General boundary: remove hypotheses
            # that do not cover the positive example
            G = [
                g for g in G
                if covers(g, example)
            ]

            # Update Specific boundary
            for i in range(len(attributes)):

                if S[i] == "Ø":
                    S[i] = example[i]

                elif S[i] != example[i]:
                    S[i] = "?"

            # Remove G hypotheses that are
            # less general than S
            G = [
                g for g in G
                if more_general_or_equal(g, S)
            ]

        # ===================================
        # NEGATIVE EXAMPLE
        # ===================================
        else:

            new_G = []

            for g in G:

                # If G already rejects negative example,
                # keep it
                if not covers(g, example):

                    new_G.append(g)

                else:
                    # Specialize G
                    specializations = specialize(
                        g,
                        example,
                        S,
                        attributes,
                        domains
                    )

                    new_G.extend(specializations)

            G = new_G

            # Remove duplicate hypotheses
            G = remove_duplicates(G)

        print("Specific Boundary (S):")
        print(S)

        print("General Boundary (G):")
        for g in G:
            print(g)

    return S, G


# -----------------------------------
# 6. Check whether hypothesis covers
#    a training example
# -----------------------------------
def covers(hypothesis, example):

    for h, x in zip(hypothesis, example):

        if h == "Ø":
            return False

        if h != "?" and h != x:
            return False

    return True


# -----------------------------------
# 7. Check whether h1 is more general
#    than or equal to h2
# -----------------------------------
def more_general_or_equal(h1, h2):

    for x, y in zip(h1, h2):

        if x == "?":
            continue

        if x == y:
            continue

        return False

    return True


# -----------------------------------
# 8. Specialize General Hypothesis
# -----------------------------------
def specialize(
        hypothesis,
        negative_example,
        S,
        attributes,
        domains):

    specializations = []

    for i in range(len(attributes)):

        # Cannot specialize if S has
        # no specific value yet
        if S[i] == "Ø":
            continue

        # Only specialize ? attributes
        if hypothesis[i] == "?":

            for value in domains[attributes[i]]:

                # Value must be different from
                # the negative example
                if value != negative_example[i]:

                    new_hypothesis = hypothesis.copy()

                    new_hypothesis[i] = value

                    # New hypothesis must still
                    # be at least as general as S
                    if more_general_or_equal(
                            new_hypothesis, S):

                        specializations.append(
                            new_hypothesis
                        )

    return specializations


# -----------------------------------
# 9. Remove duplicate hypotheses
# -----------------------------------
def remove_duplicates(hypotheses):

    unique = []

    for h in hypotheses:

        if h not in unique:
            unique.append(h)

    return unique


# -----------------------------------
# 10. Run Candidate-Elimination
# -----------------------------------
final_S, final_G = candidate_elimination(
    df,
    attributes,
    target_column,
    domains
)


# -----------------------------------
# 11. Display Final Boundaries
# -----------------------------------
print("\n========================================")
print("FINAL VERSION SPACE")
print("========================================")

print("\nMost Specific Boundary (S):")

for attribute, value in zip(
        attributes,
        final_S):

    print(f"{attribute} = {value}")


print("\nMost General Boundary (G):")

for g in final_G:
    print(g)

Original Dataset:
      Outlook Temperature Humidity    Wind PlayTennis
0       Sunny         Hot     High    Weak         No
1    Overcast        Mild   Normal  Strong         No
2    Overcast        Cool   Normal     NaN        Yes
3        Rain        Mild     High  Strong         No
4        Rain         Hot   Normal    Weak        Yes
..        ...         ...      ...     ...        ...
195      Rain        Mild     High    Weak        Yes
196  Overcast         Hot     High  Strong        Yes
197     Sunny        Mild   Normal  Strong        Yes
198      Rain        Mild     High    Weak        Yes
199      Rain        Cool     High    Weak         No

[200 rows x 5 columns]

Dataset Shape: (200, 5)

Dataset after removing missing values:
      Outlook Temperature Humidity    Wind PlayTennis
0       Sunny         Hot     High    Weak         No
1    Overcast        Mild   Normal  Strong         No
2        Rain        Mild     High  Strong         No
3        Rain         Hot   N